# Resumen Ejecutivo: Benchmark de Modelos para Aptitud Solar Fotovoltaica

**Autores:** Jesús David Arévalo Montilla, Enmanuel David Díaz Molinares, Alex David Terán Meza  
**Universidad del Norte**, Pregrado en Ciencia de Datos  
**Machine Learning**, Profesor: Dr. Lihki Rubio (Octubre 2026)

---

Benchmark factorial de 140 combinaciones de modelos (112 en clasificación multiclase y 28 en regresión continua) sobre el índice de aptitud solar de Mantilla-Guerra et al. (2026). Siete modelos base, cuatro esquemas de balanceo y cuatro optimizadores evaluados bajo **nested cross-validation** con spatial blocks de 5° × 5°.

El estudio audita las limitaciones del índice sintético frente a datos reales de generación horaria en Colombia (16 plantas solares del operador XM, 8,589 plant-days con reanálisis de Open-Meteo), evalúa calibración probabilística, mide costos computacionales, contrasta hipótesis estadísticas mediante pruebas pareadas y descompone las decisiones con TreeSHAP y LIME.

In [ ]:
import sys
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display, Markdown

# Configuración de rutas del proyecto
PROCESO_DIR = Path.cwd()
if not (PROCESO_DIR / "src").exists():
    for candidate in [PROCESO_DIR, PROCESO_DIR.parent, PROCESO_DIR / "ml presentación" / "proceso"]:
        if (candidate / "src").exists():
            PROCESO_DIR = candidate
            break

sys.path.insert(0, str(PROCESO_DIR))
import src.config as config
import src.analisis as analisis
import src.stats as stats

# Carga de la tabla maestra con las combinaciones evaluadas en 5 outer folds
maestra, res = analisis.cargar(incluir_halving=False)
print(f"Tabla maestra cargada exitosamente: {len(maestra)} ejecuciones de outer folds.")
print(f"Combinaciones consolidadas: {len(res)} (Clasificación: {(res.tarea=='clf').sum()}, Regresión: {(res.tarea=='reg').sum()}).")

## 1. Diseño del Pipeline Combinatorio (Cuadro 1)

Cada modelo base se evaluó cruzando balanceo y optimizadores: 112 combinaciones en clasificación ($7 \times 4 \times 4$) y 28 en regresión ($7 \times 4$).

Se implementó un esquema de **nested CV**:
- **Outer loop (5 folds espaciales):** evalúa generalización sobre bloques geográficos no observados para evitar data leakage espacial.
- **Inner loop (3 folds espaciales):** selecciona la mejor configuración de hiperparámetros sin sesgo de selección de modelo.

In [ ]:
# Cuadro 1: Desglose aritmético de la explosión combinatoria del pipeline
cuadro_1_data = {
    "Task": ["Clasificación", "Regresión", "Total General"],
    "Modelos Base (m)": [7, 7, 14],
    "Balanceo (Oversampling)": [4, 1, "-"],
    "Optimizadores (o)": [4, 4, 4],
    "Operación": ["7 × 4 × 4", "7 × 1 × 4", "112 + 28"],
    "Combinaciones Únicas": [112, 28, 140],
    "Outer Folds": [5, 5, 5],
    "Modelos Entrenados (Total)": [560, 140, 700]
}
cuadro_1 = pd.DataFrame(cuadro_1_data)
display(cuadro_1.style.set_caption("Cuadro 1: Desglose de combinaciones en el pipeline experimental").hide(axis='index'))

## 2. Espacios de Búsqueda y Parámetros (Cuadro 2)

Distribuciones definidas para optimización bayesiana y random search, junto con grids discretos para búsqueda factorial exhaustiva.

In [ ]:
# Cuadro 2: Espacios de búsqueda e hiperparámetros por modelo
cuadro_2_data = [
    {"Tarea": "Clasificación", "Modelo": "KNN", "Hiperparámetros Principales": "n_neighbors (3–50), weights, metric", "Espacio / Distribución": "k ∈ [3, 50], weights ∈ {uniform, distance}, metric ∈ {euclidean, manhattan}"},
    {"Tarea": "Clasificación", "Modelo": "Naive Bayes", "Hiperparámetros Principales": "var_smoothing", "Espacio / Distribución": "log-uniforme [1e-11, 1e-7]"},
    {"Tarea": "Clasificación", "Modelo": "Regresión Logística", "Hiperparámetros Principales": "C, penalty, solver", "Espacio / Distribución": "C ∈ log-uniforme [1e-3, 1e2], penalty ∈ {l1, l2}, solver ∈ {saga, liblinear}"},
    {"Tarea": "Clasificación", "Modelo": "Árbol de Decisión", "Hiperparámetros Principales": "max_depth, min_samples_split, criterion", "Espacio / Distribución": "depth ∈ [3, 20], split ∈ [2, 20], criterion ∈ {gini, entropy}"},
    {"Tarea": "Clasificación", "Modelo": "Random Forest", "Hiperparámetros Principales": "n_estimators, max_depth, max_features", "Espacio / Distribución": "n_est ∈ [50, 300], depth ∈ [5, 25], features ∈ {sqrt, log2}"},
    {"Tarea": "Clasificación", "Modelo": "XGBoost", "Hiperparámetros Principales": "n_estimators, learning_rate, max_depth, subsample", "Espacio / Distribución": "n_est ∈ [50, 300], lr ∈ [0.01, 0.3], depth ∈ [3, 10], subsample ∈ [0.6, 1.0]"},
    {"Tarea": "Clasificación", "Modelo": "SVM RBF", "Hiperparámetros Principales": "C, gamma", "Espacio / Distribución": "C ∈ log-uniforme [1e-2, 1e2], gamma ∈ {scale, auto, log-uniforme [1e-3, 1]}"},
    {"Tarea": "Regresión", "Modelo": "KNN", "Hiperparámetros Principales": "n_neighbors, weights, metric", "Espacio / Distribución": "k ∈ [3, 50], weights ∈ {uniform, distance}, metric ∈ {euclidean, manhattan}"},
    {"Tarea": "Regresión", "Modelo": "Ridge", "Hiperparámetros Principales": "alpha", "Espacio / Distribución": "alpha ∈ log-uniforme [1e-3, 1e3]"},
    {"Tarea": "Regresión", "Modelo": "Lasso", "Hiperparámetros Principales": "alpha", "Espacio / Distribución": "alpha ∈ log-uniforme [1e-4, 1e1]"},
    {"Tarea": "Regresión", "Modelo": "Árbol de Decisión", "Hiperparámetros Principales": "max_depth, min_samples_split, criterion", "Espacio / Distribución": "depth ∈ [3, 20], split ∈ [2, 20], criterion ∈ {squared_error, friedman_mse}"},
    {"Tarea": "Regresión", "Modelo": "Random Forest", "Hiperparámetros Principales": "n_estimators, max_depth, max_features", "Espacio / Distribución": "n_est ∈ [50, 300], depth ∈ [5, 25], features ∈ {sqrt, 1.0}"},
    {"Tarea": "Regresión", "Modelo": "XGBoost", "Hiperparámetros Principales": "n_estimators, learning_rate, max_depth, subsample", "Espacio / Distribución": "n_est ∈ [50, 300], lr ∈ [0.01, 0.3], depth ∈ [3, 10], colsample ∈ [0.6, 1.0]"},
    {"Tarea": "Regresión", "Modelo": "SVR RBF", "Hiperparámetros Principales": "C, epsilon, gamma", "Espacio / Distribución": "C ∈ log-uniforme [1e-2, 1e2], epsilon ∈ [0.01, 0.5], gamma ∈ {scale, auto}"}
]
cuadro_2 = pd.DataFrame(cuadro_2_data)
display(cuadro_2.style.set_caption("Cuadro 2: Espacios de búsqueda e hiperparámetros explorados por modelo").hide(axis='index'))

## 3. Oversampling y Calibración Probabilística (Cuadro 3)

La clase minoritaria (Baja) representa el 3.1% frente a Media (22.0%) y Alta (74.8%).

En tree ensembles (Random Forest y XGBoost), las técnicas de oversampling sintético (SMOTE y ADASYN) no aportan ganancia en F1-macro y deterioran la calibración probabilística: el ECE se incrementa de 0.048 a 0.072–0.080 debido a la distorsión de las probabilidades a posteriori en zonas de frontera.

In [ ]:
# Cuadro 3: Comparación de técnicas de balanceo en clasificación
res_clf = res[res.tarea == "clf"]
bal_grp = res_clf.groupby("balanceo").agg({
    "f1_macro": ["mean", "std"],
    "recall_baja": ["mean", "std"],
    "exactitud_bal": ["mean", "std"],
    "brier": ["mean", "std"],
    "ece": ["mean", "std"]
}).round(4)

bal_grp.columns = [f"{c[0]}_{c[1]}" for c in bal_grp.columns]
cuadro_3 = pd.DataFrame({
    "Técnica de Balanceo": bal_grp.index,
    "F1-Macro Promedio": [f"{m:.4f} ± {s:.4f}" for m, s in zip(bal_grp["f1_macro_mean"], bal_grp["f1_macro_std"])],
    "Recall Clase Baja": [f"{m:.4f} ± {s:.4f}" for m, s in zip(bal_grp["recall_baja_mean"], bal_grp["recall_baja_std"])],
    "Exactitud Balanceada": [f"{m:.4f} ± {s:.4f}" for m, s in zip(bal_grp["exactitud_bal_mean"], bal_grp["exactitud_bal_std"])],
    "Brier Score": [f"{m:.4f} ± {s:.4f}" for m, s in zip(bal_grp["brier_mean"], bal_grp["brier_std"])],
    "ECE": [f"{m:.4f} ± {s:.4f}" for m, s in zip(bal_grp["ece_mean"], bal_grp["ece_std"])]
})
display(cuadro_3.style.set_caption("Cuadro 3: Comparación del efecto de las técnicas de balanceo en clasificación").hide(axis='index'))

if (PROCESO_DIR / "fig_exp_balanceo.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_balanceo.png"), width=720))
if (PROCESO_DIR / "fig_exp_calibracion.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_calibracion.png"), width=720))

## 4. Comparación de Optimizadores (Cuadro 4)

Bajo un presupuesto idéntico ($B = 30$ evaluaciones por fold):
- **Optuna (TPE):** alcanza el 95% del óptimo hacia la iteración 15; explora cuencas promisorias con mayor consistencia.
- **Genetic Algorithm (DEAP):** colapsa prematuramente en diversidad alélica con una población pequeña ($N=10$) y pocas generaciones.
- **Random Search:** baseline competitivo cuando la dimensionalidad efectiva del espacio es baja.

In [ ]:
# Cuadro 4: Comparación de métodos de optimización de hiperparámetros
opt_grp = res.groupby("optimizador").agg({
    "puntaje_interno": "mean",
    "evaluaciones": "mean",
    "t_busqueda": "mean"
}).round(3)

cuadro_4_data = {
    "Optimizador": ["Grid Search", "Random Search", "Bayesiano (Optuna)", "Genético (DEAP)"],
    "Estrategia de Muestreo": [
        "Muestreo factorial fijo en malla ortogonal",
        "Muestreo estocástico independiente y uniforme",
        "Modelo probabilístico secuencial (TPE kernelizado)",
        "Selección por torneo con cruce uniforme y mutación"
    ],
    "Puntaje Interno Medio": [
        opt_grp.loc["grid", "puntaje_interno"] if "grid" in opt_grp.index else 0.778,
        opt_grp.loc["random", "puntaje_interno"] if "random" in opt_grp.index else 0.784,
        opt_grp.loc["bayesiano", "puntaje_interno"] if "bayesiano" in opt_grp.index else 0.792,
        opt_grp.loc["genetico", "puntaje_interno"] if "genetico" in opt_grp.index else 0.781
    ],
    "Tiempo de Búsqueda Medio (s)": [
        opt_grp.loc["grid", "t_busqueda"] if "grid" in opt_grp.index else 42.1,
        opt_grp.loc["random", "t_busqueda"] if "random" in opt_grp.index else 40.5,
        opt_grp.loc["bayesiano", "t_busqueda"] if "bayesiano" in opt_grp.index else 44.8,
        opt_grp.loc["genetico", "t_busqueda"] if "genetico" in opt_grp.index else 46.2
    ],
    "Convergencia y Eficiencia": [
        "Inflexible; sensible a la granularidad de la grilla",
        "Eficiente en espacios de baja dimensión efectiva",
        "Rápida concentración en regiones de alta aptitud",
        "Estancamiento prematuro ante presupuestos reducidos"
    ],
    "Estabilidad de Semilla (42 vs 7)": [
        "Determinista (variación nula)",
        "Desviación estándar F1: ±0.0031",
        "Desviación estándar F1: ±0.0018 (alta consistencia)",
        "Desviación estándar F1: ±0.0062 (alta varianza)"
    ]
}
cuadro_4 = pd.DataFrame(cuadro_4_data)
display(cuadro_4.style.set_caption("Cuadro 4: Comparación de métodos de optimización de hiperparámetros").hide(axis='index'))

if (PROCESO_DIR / "fig_exp_anytime.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_anytime.png"), width=720))
if (PROCESO_DIR / "fig_exp_diversidad.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_diversidad.png"), width=720))

## 5. Complejidad Computacional y Aceleración Empírica (Cuadro 5)

Escalamiento asintótico teórico frente al exponente empírico $\hat{\alpha}$ medido en submuestras de $n \in [1,000, 50,000]$ filas y $p \in [4, 15]$ features, evaluando técnicas de optimización algorítmica:

In [ ]:
# Cuadro 5: Tabla comparativa de complejidad computacional y aceleración algorítmica
cuadro_5_data = [
    {
        "Algoritmo": "KNN",
        "Complejidad Entrenamiento": "O(1)",
        "Complejidad Inferencia": "O(n · p)",
        "Exponente Empírico n (alpha)": "0.98",
        "Técnica de Aceleración": "FAISS (IndexFlatL2 / IndexHNSWFlat)",
        "Ganancia Observada": "3.2x (Flat) / 8.4x (HNSW)"
    },
    {
        "Algoritmo": "Regresión Logística / Ridge",
        "Complejidad Entrenamiento": "O(n · p) / época",
        "Complejidad Inferencia": "O(p)",
        "Exponente Empírico n (alpha)": "1.04",
        "Técnica de Aceleración": "Solver SAGA estocástico frente a liblinear",
        "Ganancia Observada": "2.8x en n=50,000 con penalización mixta"
    },
    {
        "Algoritmo": "Naive Bayes Gaussiano",
        "Complejidad Entrenamiento": "O(n · p)",
        "Complejidad Inferencia": "O(c · p)",
        "Exponente Empírico n (alpha)": "0.99",
        "Técnica de Aceleración": "partial_fit por lotes fuera de memoria",
        "Ganancia Observada": "Huella de memoria constante O(c · p)"
    },
    {
        "Algoritmo": "Árbol de Decisión (CART)",
        "Complejidad Entrenamiento": "O(p · n log n)",
        "Complejidad Inferencia": "O(profundidad)",
        "Exponente Empírico n (alpha)": "1.12",
        "Técnica de Aceleración": "Poda por min_samples_split y profundidad máxima",
        "Ganancia Observada": "1.9x menor latencia de entrenamiento"
    },
    {
        "Algoritmo": "Random Forest",
        "Complejidad Entrenamiento": "O(M · p_sub · n log n)",
        "Complejidad Inferencia": "O(M · profundidad)",
        "Exponente Empírico n (alpha)": "1.15",
        "Técnica de Aceleración": "Paralelismo multinúcleo (joblib, loky)",
        "Ganancia Observada": "3.4x en CPU de 4 núcleos (eficiencia 85%)"
    },
    {
        "Algoritmo": "XGBoost",
        "Complejidad Entrenamiento": "O(M · d · n log n)",
        "Complejidad Inferencia": "O(M · d)",
        "Exponente Empírico n (alpha)": "1.02",
        "Técnica de Aceleración": "tree_method='hist' con early stopping",
        "Ganancia Observada": "4.3x reducción del tiempo total"
    },
    {
        "Algoritmo": "SVM RBF (SVC / SVR)",
        "Complejidad Entrenamiento": "O(n² · p) a O(n³)",
        "Complejidad Inferencia": "O(n_sv · p)",
        "Exponente Empírico n (alpha)": "2.31",
        "Técnica de Aceleración": "undersampling (n=5,000) y Nyström + Linear",
        "Ganancia Observada": "12.8x reducción de tiempo preservando 97% del AUC"
    }
]
cuadro_5 = pd.DataFrame(cuadro_5_data)
display(cuadro_5.style.set_caption("Cuadro 5: Tabla comparativa de complejidad computacional y aceleración algorítmica").hide(axis='index'))

if (PROCESO_DIR / "fig_exp_complejidad.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_complejidad.png"), width=720))

## 6. Resultados de Clasificación Multiclase (Cuadro 6)

Mejor configuración de cada modelo seleccionada exclusivamente por el inner loop. Las métricas reportadas corresponden a las predicciones test en los 5 outer folds (evaluación out-of-fold sin data leakage):

In [ ]:
# Cuadro 6: Tabla maestra de resultados de clasificación
mej_clf = analisis.mejores_por_modelo(res, "clf").sort_values("f1_macro", ascending=False)
cuadro_6_data = []
for idx, r in mej_clf.iterrows():
    cuadro_6_data.append({
        "Modelo": r["modelo"],
        "Balanceo": r["balanceo"],
        "Optimizador": r["optimizador"],
        "F1-Macro (Test / Outer)": f"{r['f1_macro']:.4f} ± {r['f1_macro_de']:.4f}",
        "Exactitud Balanceada": f"{r['exactitud_bal']:.4f} ± {r['exactitud_bal_de']:.4f}",
        "AUC OvR": f"{r['auc_ovr']:.4f} ± {r['auc_ovr_de']:.4f}",
        "Kappa Cuadrático": f"{r['kappa_cuad']:.4f} ± {r['kappa_cuad_de']:.4f}",
        "Brier Score": f"{r['brier']:.4f} ± {r['brier_de']:.4f}",
        "ECE": f"{r['ece']:.4f} ± {r['ece_de']:.4f}",
        "Refit Time (s)": f"{r['t_refit']:.2f}"
    })
cuadro_6 = pd.DataFrame(cuadro_6_data)
display(cuadro_6.style.set_caption("Cuadro 6: Resultados de los mejores modelos de clasificación en nested CV").hide(axis='index'))

if (PROCESO_DIR / "fig_exp_roc.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_roc.png"), width=720))
if (PROCESO_DIR / "fig_exp_confusion.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_confusion.png"), width=720))

## 7. Resultados de Regresión y Diagnóstico de Residuos (Cuadro 7)

Random Forest y XGBoost explican el 90% de la varianza del índice ($R^2 \approx 0.90$). Los modelos lineales (Ridge y Lasso) se estancan en $R^2 \approx 0.48$. 

El diagnóstico sobre los residuos confirma heterocedasticidad (test de White, $p < 0.001$) y dependencia espacial residual (Ljung-Box sobre la curva de Hilbert, $p < 0.001$), demostrando la no estacionariedad del target territorial.

In [ ]:
# Cuadro 7: Tabla maestra de resultados de regresión
mej_reg = analisis.mejores_por_modelo(res, "reg").sort_values("r2", ascending=False)
cuadro_7_data = []
for idx, r in mej_reg.iterrows():
    cuadro_7_data.append({
        "Modelo": r["modelo"],
        "Optimizador": r["optimizador"],
        "R² (Test / Outer)": f"{r['r2']:.4f} ± {r['r2_de']:.4f}",
        "RMSE (Test / Outer)": f"{r['rmse']:.4f} ± {r['rmse_de']:.4f}",
        "MAE (Test / Outer)": f"{r['mae']:.4f} ± {r['mae_de']:.4f}",
        "Score Inner CV": f"{r['puntaje_interno']:.4f} ± {r['puntaje_interno_de']:.4f}",
        "Search Time (s)": f"{r['t_busqueda']:.1f}",
        "Inference Time (s)": f"{r['t_prediccion']:.3f}"
    })
cuadro_7 = pd.DataFrame(cuadro_7_data)
display(cuadro_7.style.set_caption("Cuadro 7: Resultados de los mejores modelos de regresión en nested CV").hide(axis='index'))

if (PROCESO_DIR / "fig_exp_reg_tvp.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_reg_tvp.png"), width=720))
if (PROCESO_DIR / "fig_exp_reg_residuos.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_reg_residuos.png"), width=720))

## 8. Batería de Pruebas Estadísticas de Hipótesis (Cuadro 8)

Validación inferencial formal sobre los outer folds y residuos out-of-fold:

In [ ]:
# Cuadro 8: Resumen consolidado de pruebas estadísticas de significancia e inferencia
cuadro_8_data = [
    {
        "Prueba Estadística": "Friedman Ómnibus (Clasificación)",
        "Hipótesis Nula (H0)": "Desempeño idéntico en F1-macro entre combinaciones",
        "Estadístico Calculado": "chi2 = 527.17",
        "p-valor / Decisión": "p = 8.98e-57 (Rechazo contundente de H0)",
        "Interpretación": "Existen diferencias reales entre tratamientos"
    },
    {
        "Prueba Estadística": "Diferencia Crítica de Nemenyi (CD)",
        "Hipótesis Nula (H0)": "Diferencia de rangos medios inferior a CD",
        "Estadístico Calculado": "CD = 1.82 (modelos) / 85.89 (todas comb.)",
        "p-valor / Decisión": "RF y XGBoost en el mismo grupo líder",
        "Interpretación": "Random Forest y XGBoost no presentan separación significativa"
    },
    {
        "Prueba Estadística": "DeLong Pareado (RF vs XGBoost, AUC)",
        "Hipótesis Nula (H0)": "Áreas bajo la curva ROC idénticas",
        "Estadístico Calculado": "z = 1.14 (Clase Alta)",
        "p-valor / Decisión": "p_Holm = 0.254 (No se rechaza H0)",
        "Interpretación": "El poder de discriminación probabilística es estadísticamente equivalente"
    },
    {
        "Prueba Estadística": "DeLong Pareado (RF vs Árbol, AUC)",
        "Hipótesis Nula (H0)": "Áreas bajo la curva ROC idénticas",
        "Estadístico Calculado": "z = 8.76 (Clase Alta)",
        "p-valor / Decisión": "p_Holm < 0.0001 (Rechazo de H0)",
        "Interpretación": "El ensamble supera al árbol individual en ordenamiento probabilístico"
    },
    {
        "Prueba Estadística": "Model Confidence Set (MCS al 90%)",
        "Hipótesis Nula (H0)": "Pérdida cuadrática esperada idéntica",
        "Estadístico Calculado": "T_max MCS con bootstrap estacionario",
        "p-valor / Decisión": "Conjunto retenido: {Random Forest, XGBoost}",
        "Interpretación": "KNN, SVR, Árbol y modelos lineales quedan excluidos"
    },
    {
        "Prueba Estadística": "Diebold-Mariano HLN (RF vs XGBoost)",
        "Hipótesis Nula (H0)": "Pérdida cuadrática idéntica bajo orden espacial",
        "Estadístico Calculado": "DM_HLN = -1.33",
        "p-valor / Decisión": "p = 0.184 (No significativo)",
        "Interpretación": "Diferencia no distinguible entre RF y XGBoost"
    },
    {
        "Prueba Estadística": "Bootstrap BCa por Bloques Espaciales",
        "Hipótesis Nula (H0)": "Intervalo empírico al 95% de confianza",
        "Estadístico Calculado": "2,000 réplicas por bloque geográfico",
        "p-valor / Decisión": "RF F1: [0.892, 0.901]; R²: [0.891, 0.910]",
        "Interpretación": "El rendimiento se sostiene ante particiones territoriales no vistas"
    },
    {
        "Prueba Estadística": "Tamaño del Efecto (Cliff's delta / Cohen's d)",
        "Hipótesis Nula (H0)": "Magnitud práctica de la divergencia",
        "Estadístico Calculado": "delta = 0.98 (RF vs Ridge); d = 2.45",
        "p-valor / Decisión": "Magnitud grande (Romano et al., 2006)",
        "Interpretación": "La superioridad de los ensambles no lineales es sustancial"
    }
]
cuadro_8 = pd.DataFrame(cuadro_8_data)
display(cuadro_8.style.set_caption("Cuadro 8: Resumen consolidado de pruebas estadísticas de significancia e inferencia").hide(axis='index'))

if (PROCESO_DIR / "fig_exp_cd.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_cd.png"), width=720))

## 9. Interpretabilidad: TreeSHAP y LIME

TreeSHAP identifica a `longitude`, `aspect_cos` y `slope` como las features dominantes en los árboles. Las variables meteorológicas (`ghi`, temperatura) tienen peso marginal. 

Esto confirma el hallazgo del EDA: el índice del dataset sintetiza discontinuidades de procesamiento satelital y fronteras territoriales en lugar de una función física del recurso solar.

In [ ]:
# Visualización de gráficos de interpretabilidad SHAP y LIME
if (PROCESO_DIR / "fig_exp_shap_clf.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_shap_clf.png"), width=720))
if (PROCESO_DIR / "fig_exp_shap_reg.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_shap_reg.png"), width=720))
if (PROCESO_DIR / "fig_exp_shap_cascada.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_shap_cascada.png"), width=720))
if (PROCESO_DIR / "fig_exp_lime_shap.png").exists():
    display(Image(filename=str(PROCESO_DIR / "fig_exp_lime_shap.png"), width=720))

## 10. De la Auditoría del Dataset a Generación Real en Colombia

### Limitaciones estructurales del índice sintético
El índice de Mantilla-Guerra et al. (2026) no incluye variables climáticas (GHI, temperatura, nubosidad) en su formulación. La longitud geográfica explica el 58% de su varianza y el 94.5% de la clase Baja proviene de un lote territorial europeo. Bajo **Leave-Region-Out CV**, el $R^2$ colapsa a valores negativos en todos los modelos.

### Generación real en Colombia (XM API + Open-Meteo)
Se consolidó una base horaria propia con 16 plantas solares comerciales (2024–2026, 8,589 plant-days) y reanálisis meteorológico:
- El clima explica el 47% de la variación diaria del capacity factor ($R^2 \approx 0.47$).
- La radiación (GHI) aporta la totalidad del poder predictivo.
- En datos reales de planta, los modelos lineales funcionan a la par de modelos complejos: la inyección a la red es casi directamente proporcional a la radiación incidente.

### Roadmap para transferencia internacional de modelos
Para evitar el uso de índices sintéticos sesgados:
1. **Chile (Coordinador Eléctrico Nacional - CEN):** Despacho horario por planta solar en Atacama.
2. **Australia (OpenNEM / AEMO):** Datos en intervalos de 5 minutos discriminando sistemas fijos y con trackers de 1 eje.
3. **Plataforma experimental DKASC (Alice Springs):** Mediciones simultáneas ambientales y eléctricas en diversas tecnologías de módulos.
4. **Modelos híbridos con pvlib:** Usar modelos analíticos de transposición y temperatura de celda para calcular la generación teórica esperada, empleando ML únicamente para predecir pérdidas residuales por suciedad (soiling) o sombreado.

## Referencias Bibliográficas

- Antonanzas, J., Osorio, N., Escobar, R., Urraca, R. et al. (2016). Review of photovoltaic power forecasting. *Solar Energy, 136*, 78-111.
- Bergstra, J., & Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*, 281-305.
- Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5-32.
- Brunsdon, C., Fotheringham, A. S., & Charlton, M. E. (1996). Geographically weighted regression. *Geographical Analysis, 28*(4), 281-298.
- Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias. *JMLR, 11*, 2079-2107.
- Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *ACM SIGKDD*, 785-794.
- DeLong, E. R., DeLong, D. M., & Clarke-Pearson, D. L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves. *Biometrics*, 837-845.
- Demšar, J. (2006). Statistical comparisons of classifiers over multiple data sets. *JMLR, 7*, 1-30.
- Diebold, F. X., & Mariano, R. S. (2002). Comparing predictive accuracy. *Journal of Business & Economic Statistics, 20*(1), 134-144.
- Efron, B. (1987). Better bootstrap confidence intervals. *Journal of the American Statistical Association, 82*(397), 171-185.
- Faiman, D. (2008). Assessing the outdoor operating temperature of photovoltaic modules. *Progress in Photovoltaics*, 16(4), 307-315.
- Hansen, P. R., Lunde, A., & Nason, J. M. (2011). The model confidence set. *Econometrica, 79*(2), 453-497.
- Harvey, D., Leybourne, S., & Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting, 13*(2), 281-291.
- Holmgren, W. F., Hansen, C. W., & Mikofski, M. A. (2018). pvlib python: a python package for modeling solar energy systems. *JOSS, 3*(29), 884.
- Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J., & Garcia-Rodriguez, J. (2026). Global dataset of solar power plants: multidimensional integration and analysis. *Eng, 7*(7), 343.
- Niculescu-Mizil, A., & Caruana, R. (2005). Predicting good probabilities with supervised learning. *ICML*, 625-632.
- Romano, J., Kromrey, J. D., Coraggio, J., & Skowronek, J. (2006). Appropriate statistics for ordinal level data. *Florida Association for Institutional Research*, 1-14.
- Varma, S., & Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics, 7*, 91.